# EXP-008 판정 근거: 안건 항목 법률명 조건

EXP-004 대비 single-doc·multi-doc 지표, 유형별 차이, 최다 일치 필터 집합의 변화, 검색 시간을 보고 채택 여부를 정한다. 질의 원문은 출력하지 않는다.

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from rag.index import load_cfg
from rag.search import condition_setup

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
BASE_CFG = load_cfg(['+exp=exp004', 'index.scope=full', 'search.queries=multidoc'])
CAND_CFG = load_cfg(['+exp=exp008', 'index.scope=full', 'search.queries=multidoc'])
BASE_RUN = RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-multidoc.csv'
CAND_RUN = RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-law-multidoc.csv'
SEARCH_SECONDS = {'EXP-004': 47, 'EXP-008': 51}  # 같은 PC, dev-full 프로세스 전체, 1회

## single-doc (판정 1단계)

In [13]:
single = json.loads((RUNS / 'exp008-single-compare.json').read_text(encoding='utf-8'))
print('순위 파일 해시 같음', single['base']['run_sha256'] == single['cand']['run_sha256'])
print('Recall@5 차이', single['diff'], single['ci'], single['verdict'])

순위 파일 해시 같음 True
Recall@5 차이 0.0 [0.0, 0.0] 보류


## multi-doc (판정 2단계)

In [14]:
multi = json.loads((RUNS / 'exp008-multi-compare.json').read_text(encoding='utf-8'))
print(pd.DataFrame({'EXP-004': multi['base']['metrics'],
                    'EXP-008': multi['cand']['metrics']}).round(4))
d = pd.DataFrame(multi['diffs'])
d = d[d['metric'].isin(['recall@5', 'complete@5'])].copy()
d['diff_%p'] = (d['diff'] * 100).round(2)
d['ci_%p'] = d['ci'].map(lambda c: [round(x * 100, 2) for x in c])
print(d[['group', 'value', 'n', 'metric', 'diff_%p', 'ci_%p']].to_string(index=False))

             EXP-004  EXP-008
recall@5      0.9361   0.9527
complete@5    0.8717   0.9050
hit@5         0.9883   0.9917
recall@10     0.9597   0.9720
complete@10   0.9217   0.9483
hit@10        0.9917   0.9950
group      value   n     metric  diff_%p         ci_%p
  all        all 600   recall@5     1.65  [0.91, 2.49]
  all        all 600 complete@5     3.33   [2.0, 4.83]
 type       conf 233   recall@5     0.11   [0.0, 0.32]
 type       conf 233 complete@5     0.43   [0.0, 1.29]
 type        law 152   recall@5     6.36  [3.51, 9.48]
 type        law 152 complete@5    12.50 [7.24, 18.42]
 type questioner 215   recall@5     0.00    [0.0, 0.0]
 type questioner 215 complete@5     0.00    [0.0, 0.0]


## 최다 일치 필터 집합의 변화

`rag.search`와 같은 규칙(맞은 필드 수가 최대이고 `min_fields` 이상인 문서)으로 기준·후보의 필터 집합을 만들어, 크기와 그 안의 정답 문서 수를 유형별로 센다.

In [19]:
with (ROOT / BASE_CFG.paths.index_dir / 'chunks.jsonl').open(encoding='utf-8') as f:
    chunk_doc_ids = np.array([json.loads(line)['doc_id'] for line in f])
setups = {name: condition_setup(cfg, chunk_doc_ids) for name, cfg in
          [('base', BASE_CFG), ('cand', CAND_CFG)]}
min_fields = BASE_CFG.retriever.conditions.min_fields
with (ROOT / BASE_CFG.paths.multidoc_frozen).open(encoding='utf-8') as f:
    frozen = [json.loads(line) for line in f]
print(len(frozen), 'queries')

600 queries


In [20]:
def filter_set(setup, qid):
    names, _, index, conds = setup
    counts = index.match_counts(conds[qid])
    if not counts or max(counts.values()) < min_fields:
        return set()
    top = max(counts.values())
    return {names[i] for i, c in counts.items() if c == top}

rows = []
for q in frozen:
    b, c = filter_set(setups['base'], q['qid']), filter_set(setups['cand'], q['qid'])
    gold = set(q['gold_doc_ids'])
    rows.append({'type': q['type'], 'changed': b != c, 'base_size': len(b), 'cand_size': len(c),
                 'base_gold': len(gold & b), 'cand_gold': len(gold & c)})
fs = pd.DataFrame(rows)
print(fs[fs['changed']].groupby('type')[['base_size', 'cand_size', 'base_gold', 'cand_gold']]
      .agg(['count', 'sum']).iloc[:, [0, 1, 3, 5, 7]])

           base_size       cand_size base_gold cand_gold
               count   sum       sum       sum       sum
type                                                    
conf              21   255       107        49        49
law               58  1067       263       154       156
questioner         1     2         1         2         1


## 질의별 증감 (multi-doc Recall@5·Complete@5)

In [21]:
def top5(path):
    run = pd.read_csv(path, dtype=str)
    run['rank'] = run['rank'].astype(int)
    return run[run['rank'] <= 5].groupby('qid')['doc_id'].apply(set)

b5, c5 = top5(BASE_RUN), top5(CAND_RUN)
per = []
for q in frozen:
    gold = set(q['gold_doc_ids'])
    rb = len(gold & b5[q['qid']]) / len(gold)
    rc = len(gold & c5[q['qid']]) / len(gold)
    per.append({'type': q['type'], 'up': rc > rb, 'down': rc < rb,
                'complete_gain': rc == 1 and rb < 1, 'complete_loss': rb == 1 and rc < 1})
print(pd.DataFrame(per).groupby('type').sum())

            up  down  complete_gain  complete_loss
type                                              
conf         1     0              1              0
law         22     1             20              1
questioner   0     0              0              0


## 검색 시간

In [22]:
print(SEARCH_SECONDS, f"+{SEARCH_SECONDS['EXP-008'] / SEARCH_SECONDS['EXP-004'] - 1:.0%}")

{'EXP-004': 47, 'EXP-008': 51} +9%
